In [1]:
# 모듈 import
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError
from typing import Annotated, Literal
from annotated_types import Gt

In [2]:
# class 선언부
class User(BaseModel):
    id : int
    name : str = "John Doe"
    signup_ts : datetime | None
    tastes : dict[str, PositiveInt] #Positivelnt : 양의 정수

In [3]:
# data
external_data = {
    'id' : 123,
    'signup_ts' : '2019-06-01 12:22',
    'tastes' : {
        'wine' : 9,
        b'cheese' : 7,  #b: bytes데이터
        'cabbage' : '1',
        },
}

In [4]:
# user object 생성
user = User(**external_data) #** :  항목별로, 해당 이름에 맞는 값을 넣음

In [5]:
print(user.id)
print(user.signup_ts)
print(user.tastes) # 자동 형변환
print(user.model_dump()) #.model_dump : Pydantic 데이터를 dict로 꺼내는 기능

123
2019-06-01 12:22:00
{'wine': 9, 'cheese': 7, 'cabbage': 1}
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [7]:
external_data2 = {'id': 'not an int', 'tastes': {}}

In [8]:
try:
    User(**external_data2)
except ValidationError as e: #에러코드 별칭은 대부분 e를 씀
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [9]:
class Fruit(BaseModel):
    name : str
    color : Literal['red', 'green'] #Literal: 지정된 값만 사용
    weight : Annotated[float, Gt(0)] #Annotated: 값의 조건을 붙임, Gt: Gt의 지정값보다 커야함
    bazam : dict[str, list[tuple[int, bool, float]]] #tuple: 순서가 정해져있고 변경할 수 없음

In [10]:
print(
    Fruit(
        name = 'Apple',
        color = 'red',
        weight = 1,
        bazam = {'foobar': [(1, True, 0.1)]} #tuple은 소괄호
    )
)

name='Apple' color='red' weight=1.0 bazam={'foobar': [(1, True, 0.1)]}


In [11]:
class Meeting(BaseModel):
    when : datetime
    where : bytes
    why : str = "No idea"

In [12]:
m = Meeting(when = '2020-01-01T12:00', where = 'home')

In [13]:
print(m.model_dump)

#exclude_unset=True: 내가 설정하지 않은 항목 빼기
print(m.model_dump(exclude_unset=True))

#exclude={'지정항목'}: 지정한 항목 빼기, mode='json': json로 표현 가능한 형태로 바꾼 dict
print(m.model_dump(exclude={'where'}, mode='json'))

#exclude_defaults=True): 기본값과 같은 항목 빼기, model_dump_json:json 문자열
print(m.model_dump_json(exclude_defaults=True))

<bound method BaseModel.model_dump of Meeting(when=datetime.datetime(2020, 1, 1, 12, 0), where=b'home', why='No idea')>
{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}
{'when': '2020-01-01T12:00:00', 'why': 'No idea'}
{"when":"2020-01-01T12:00:00","where":"home"}
